# Lesson 18 — Decomposition Methods

## Learning objectives

1. Explain the structure that makes a problem decomposable.
2. Implement Benders decomposition for a 2-stage stochastic LP.
3. Recognize Dantzig-Wolfe decomposition and column generation.
4. Recognize Lagrangian relaxation and the link to dual decomposition.

## 1. Block-angular structure

Many large LPs / MILPs have the structure

$$
\begin{aligned}
\min \quad & c^\top x + \sum_{s} d_s^\top y_s \\
\text{s.t.} \quad & A_s x + B_s y_s = b_s, \quad s = 1, \dots, S \\
& T x = h \quad \text{(complicating constraint on } x\text{ alone)}.
\end{aligned}
$$

If we fix $x$, the problem decouples across $s$. Decomposition methods exploit this — solve smaller subproblems, coordinate via a *master*.

## 2. Benders decomposition

For 2-stage stochastic LPs / mixed-integer first stage:

- **Master:** $\min c^\top x + \eta$ s.t. (cuts so far), $x \in \mathcal{X}$.
- **Subproblem (per scenario):** given $\hat x$, solve $\min d_s^\top y_s$ s.t. $B_s y_s = b_s - A_s \hat x, y_s \ge 0$.
- **Cuts:** if subproblem feasible with dual $\pi_s$, add *optimality cut* $\eta \ge \pi_s^\top (b_s - A_s x)$. If infeasible with dual ray $\rho_s$, add *feasibility cut* $\rho_s^\top (b_s - A_s x) \le 0$.

Iterate until master and sum-of-subproblem objectives match. {cite:t}`Benders1962` is the original; {cite:p}`Birge2011` is the modern stochastic-programming treatment.

## 3. Dantzig-Wolfe / column generation

For block-angular *with linking constraints*, Dantzig-Wolfe expresses each block's feasible polytope as a convex combination of extreme points (a Minkowski sum). The master selects which combinations to use; subproblems generate new extreme points (columns) only when needed (pricing). {cite:p}`DantzigWolfe1960`.

Cutting stock and crew scheduling are textbook examples.

## 4. Lagrangian relaxation

Move the complicating *equality* constraint $Tx = h$ into the objective with multipliers $\mu$ (free, since the constraint is an equality — we follow the convention from Lesson 9: $\lambda$ for inequalities, $\mu$ for equalities):

$$L(\mu) = \min_x c^\top x + \mu^\top (Tx - h).$$

For each fixed $\mu$ this is decomposable (or simpler). The Lagrangian dual $\max_\mu L(\mu)$ provides a lower bound on the primal optimum. Subgradient methods or bundle methods solve the outer maximization. (For an *inequality* complicating constraint $Tx \le h$, use $\lambda \ge 0$ instead.)

Often used as a *bound* inside B&B for hard MILPs (e.g., generalized assignment). {cite:p}`Wolsey1998` Ch 10.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do
from discopt.modeling import Model

# Toy 2-stage stochastic LP, 3 scenarios (production w/ uncertain demand)
S = 3
demand = np.array([80, 100, 120])  # scenarios
prob   = np.array([0.3, 0.4, 0.3])

# First stage: x = production capacity (cost 5/unit)
# Second stage (per scenario): y = sales (revenue 9/unit), with y <= x and y <= demand_s
def deterministic_eq():
    m = Model()
    x = m.continuous("x", lb=0)
    y = [m.continuous(f"y{s}", lb=0) for s in range(S)]
    m.minimize(5 * x - sum(prob[s] * 9 * y[s] for s in range(S)))
    for s in range(S):
        m.subject_to(y[s] <= x); m.subject_to(y[s] <= demand[s])
    return m.solve()

print("EQ form:", deterministic_eq().objective)
# A Benders implementation would split this into a master(x, eta) and S subproblems(y_s).


## References
{cite:p}`Benders1962,DantzigWolfe1960,Birge2011,Wolsey1998`. For a modern algorithmic perspective, {cite:p}`Achterberg2007` discusses how solvers integrate decomposition with B&B.